# Retinal image testing: Jev ? RETFound

An explained, runnable companion to the existing **local server** and `/dataset-tests` page.

**Goal:** check whether an image is a retinal fundus photograph, classify accepted images into five diabetic retinopathy (DR) grades, and compare the predicted grade with its dataset reference label.

This notebook contains **15 existing test images: five IDRiD, five Messidor2, five DDR**, one image per grade in each dataset. It does not download data, train a model, change weights, or load a second RETFound model. Inference runs through the server's existing implementation.

Run cells in order. Previewing is local. The inference switches default to `False`; enabling them sends screening copies to Jev through OpenRouter and may incur charges. No fresh predictions are included in this notebook.


## 1. Start the local server and choose the kernel

Use the **ml-homework** Python kernel. The existing environment already contains JupyterLab, IPython, Pillow and notebook tools; no installation is needed.

From a PowerShell terminal in the RETFound project, start Jupyter:

```powershell
& "$env:USERPROFILE\.conda\envs\ml-homework\python.exe" -m jupyterlab
```

Open `examples/RETFound_Jev_15_Image_Test_Pipeline.ipynb`.
In a separate terminal, start the server with its existing key:

```powershell
& "$env:USERPROFILE\.conda\envs\ml-homework\python.exe" -B scripts/serve_web.py
```

If the server needs a new key, stop the old server and use the hidden prompt:

```powershell
& "$env:USERPROFILE\.conda\envs\ml-homework\python.exe" -B scripts/diagnose_jev.py --new-key --start-server
```

The diagnostic makes billed model checks after authentication. Never put your key in this notebook. The key stays on the backend. The notebook's HTTP requests go to `http://127.0.0.1:8000` without a browser Origin header; browser CORS settings are not needed.


In [ ]:
from pathlib import Path
import sys, json, hashlib, html, io, socket, uuid
from datetime import datetime, timezone
from urllib import request, error, parse
from IPython.display import display, HTML
from PIL import Image

# Locate the project from Jupyter's current directory or a parent.
# If you launch Jupyter elsewhere, set PROJECT_ROOT to the explicit project path.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'retfound_platform').is_dir() and (p / 'docs/PROGRESS.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Set PROJECT_ROOT to the RETFound project directory, then rerun.')
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from retfound_platform.api.demo_samples import DemoSamples

DEMO_ROOT = PROJECT_ROOT / 'upload_test_images/huggingface15-20261030'
BASE_URL = 'http://127.0.0.1:8000'
RUN_ONE = False       # Change to True only when you want a real Jev + RETFound request.
RUN_ALL = False       # Change to True for up to 15 sequential real requests.
SAMPLE_ID = 'IDRID-0'
REQUEST_TIMEOUT = 180
results = {}         # This session only. Re-running this cell resets the notebook's results.
outcome_unknown = False
server_verified = False
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)
print('Live inference:', {'one': RUN_ONE, 'all': RUN_ALL})


## 2. Follow the pipeline

Jev's output is a **modality admission decision**, not the DR grade. Even grade 0 (No DR) must pass the fundus check. Server errors and timeouts are not decision 0.


In [ ]:
display(HTML("""
<div style="font-family:Arial;max-width:950px;line-height:1.6">
<div style="padding:12px;background:#e9f5f2;border:1px solid #15796b;border-radius:8px">
<b>1 ? Fixed test image ID</b> ? server resolves manifest entry ? verifies SHA-256 ? decodes original bytes</div>
<div style="text-align:center">?</div>
<div style="padding:12px;background:#eef3ff;border:1px solid #5276aa;border-radius:8px">
<b>2 ? Jev modality check</b><br>Backend HTTP POST ? OpenRouter decisions API ? typesafe/jev-1.14<br>
A compatible screening copy is prepared when needed. Original RETFound input bytes are retained.</div>
<table style="width:100%;margin:10px 0;border-collapse:collapse"><tr>
<td style="padding:12px;border:1px solid #bf655e"><b>0 ? Not identified as fundus</b><br>Reject (HTTP 422). Stop before RETFound.</td>
<td style="padding:12px;border:1px solid #b39440"><b>Error / unavailable</b><br>HTTP 503. Block RETFound. No binary decision inferred.</td>
<td style="padding:12px;border:1px solid #15796b"><b>1 ? Identified as fundus</b><br>Continue with original image bytes.</td>
</tr></table>
<div style="text-align:center">Only decision 1 ?</div>
<div style="padding:12px;background:#e9f5f2;border:1px solid #15796b;border-radius:8px">
<b>3 ? Existing RETFound classifier</b><br>RGB ? bicubic shorter edge 256 ? center crop 224 ? tensor ? ImageNet normalization<br>
APTOS checkpoint ? five logits ? softmax scores ? argmax DR grade 0?4</div>
<div style="text-align:center">?</div>
<div style="padding:12px;background:#f5f5f5;border:1px solid #777;border-radius:8px">
<b>4 ? Backend comparison</b> ? join dataset reference label after inference ? match / mismatch ? notebook displays and optionally exports</div>
</div>
"""))


## 3. Understand the sources and labels

| Dataset | Source | Selection and reference labels |
|---|---|---|
| IDRiD | [amin-nejad/idrid-disease-grading](https://huggingface.co/datasets/amin-nejad/idrid-disease-grading) | First test row per grade within the first 100 rows; explicit ClassLabel mapping. |
| Messidor2 | [OctoMed/Messidor2](https://huggingface.co/datasets/OctoMed/Messidor2) | First test row per grade within the first 100 rows; `answer` A?E maps to 0?4. Generated responses are not reference labels. |
| DDR | [ctmedtech/DDR-dataset](https://huggingface.co/datasets/ctmedtech/DDR-dataset/tree/main/DR_grading/test) | First sorted test filename per grade, joined to `DR_grading/test.txt`. Extra grade 5 is excluded. |

| DR grade | Label |
|---|---|
| 0 | No DR |
| 1 | Mild |
| 2 | Moderate |
| 3 | Severe |
| 4 | Proliferative |

These are dataset-supplied reference labels, not independent clinical reviews. IDRiD and Messidor2 files are Hugging Face viewer-exported images; equivalence to original embedded bytes is unverified. DDR files use pinned repository bytes. Source revisions and image hashes are preserved in the local manifest.

This is a deterministic **balanced demonstration**, not a random representative evaluation. Broader split overlap, patient overlap, near duplicates and checkpoint-training independence are unverified.


In [ ]:
demo = DemoSamples(DEMO_ROOT)
samples = demo.catalog()  # Validates manifest identity, mapping, counts and root containment.
by_id = {s['id']: s for s in samples}
for sample in samples:
    _, payload = demo.load(sample['id'])  # Checks bounded read and exact-byte SHA-256.
    with Image.open(io.BytesIO(payload)) as image:
        image.load()
print(f'Verified {len(samples)} local images; no API calls or model inference.')

def table(headers, rows):
    esc = lambda x: html.escape(str(x))
    body = ''.join('<tr>' + ''.join('<td style="padding:6px;border:1px solid #ddd">' + esc(v) + '</td>' for v in row) + '</tr>' for row in rows)
    display(HTML('<table style="border-collapse:collapse"><tr>' + ''.join('<th style="padding:6px">' + esc(h) + '</th>' for h in headers) + '</tr>' + body + '</table>'))

table(['Sample ID', 'Split', 'Reference grade', 'Reference label', 'SHA-256 prefix'],
      [(s['id'], s['split'], s['true_grade'], s['true_label'], s['sha256'][:16]) for s in samples])


## 4. Preview all 15 images

The following contact sheets are built from local files only. Preview resizing affects display only; it does not modify source files or model preprocessing. Captions show **reference**, not predicted, grades.


In [ ]:
import base64
for dataset in ('IDRID', 'MESSIDOR2', 'DDR'):
    cards = []
    for s in samples:
        if s['dataset'] != dataset:
            continue
        _, payload = demo.load(s['id'])
        with Image.open(io.BytesIO(payload)) as image:
            thumb = image.convert('RGB')
            thumb.thumbnail((240, 200))
            buffer = io.BytesIO()
            thumb.save(buffer, format='JPEG', quality=85)
        uri = base64.b64encode(buffer.getvalue()).decode('ascii')
        caption = html.escape(f"{s['id']} | {s['true_label']}")
        cards.append(f'<figure style="margin:6px;width:240px"><img alt="{caption}" src="data:image/jpeg;base64,{uri}"><figcaption>{caption}</figcaption></figure>')
    display(HTML('<h3>' + dataset + '</h3><div style="display:flex;flex-wrap:wrap">' + ''.join(cards) + '</div>'))


## 5. Check the server without invoking Jev

The notebook uses the same catalog, fixed image IDs and prediction route as the web page:

- `GET /api/health`, `/api/model`, `/api/demo`: read-only readiness, model identity and catalog.
- `POST /api/demo/{sample_id}/predict`: real screening and classification request, with no label or image supplied by this notebook.

The server retrieves the verified image itself. Reference labels are joined after inference and never supplied to Jev or RETFound as hints. The key and OpenRouter payload logic remain in `retfound_platform/inference/fundus_gate.py`; shared inference and comparison run in `retfound_platform/api/app.py`.


In [ ]:
# Loopback only; bypass system proxies and disallow HTTP redirects.
parts = parse.urlsplit(BASE_URL)
if parts.scheme != 'http' or parts.hostname not in ('127.0.0.1', 'localhost', '::1') or parts.path not in ('', '/') or parts.username or parts.password or parts.query or parts.fragment:
    raise ValueError('BASE_URL must identify the local HTTP server only.')
BASE_URL = BASE_URL.rstrip('/')
class NoRedirect(request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None
opener = request.build_opener(request.ProxyHandler({}), NoRedirect())

def api(path, method='GET'):
    req = request.Request(BASE_URL + path, data=b'' if method == 'POST' else None,
                          method=method, headers={'Accept': 'application/json'})
    try:
        with opener.open(req, timeout=REQUEST_TIMEOUT) as response:
            status, raw = response.status, response.read(2_000_001)
    except error.HTTPError as exc:
        status, raw = exc.code, exc.read(2_000_001)
    if len(raw) > 2_000_000:
        raise ValueError('Unexpectedly large server response.')
    return status, json.loads(raw)

server_verified = False
try:
    status, health = api('/api/health')
    if status != 200 or health.get('status') != 'ready':
        raise RuntimeError(f'Server is not ready (HTTP {status}).')
    status, model_identity = api('/api/model')
    if status != 200 or model_identity.get('name') != 'RETFound MAE':
        raise RuntimeError('Unexpected model service.')
    status, catalog = api('/api/demo')
    remote = {s['id']: s for s in catalog.get('samples', [])}
    if status != 200 or len(catalog.get('samples', [])) != 15 or set(remote) != set(by_id):
        raise RuntimeError('Server sample catalog differs from this notebook.')
    for sid, sample in by_id.items():
        if any(remote[sid][key] != sample[key] for key in ('sha256', 'true_grade', 'true_label', 'dataset', 'split')):
            raise RuntimeError('Server reference or image identity differs: ' + sid)
    server_verified = True
    print('Server ready. Catalog agrees with all 15 local images.')
    display(HTML('<pre>' + html.escape(json.dumps(model_identity, indent=2)) + '</pre>'))
except (OSError, ValueError, RuntimeError) as exc:
    print('Live testing disabled until this check succeeds:', str(exc))
    print('You can still view all local images and explanations.')


## 6. Test one image

Change `RUN_ONE = True` in the settings cell and choose `SAMPLE_ID`, then rerun cells in order. One POST can invoke a paid Jev request. It reaches RETFound only after decision 1. An HTTP 422 with `not_retinal_fundus` records decision 0; HTTP 503 is an unavailable check, not a rejection.

The notebook keeps complete server responses and checks image identity and binary admission on successful responses. Grade agreement is taken from the backend; this notebook does not implement a second classifier or metric engine. Rerunning a completed sample is skipped; restarting the settings cell clears this session and may lead to new paid requests.


In [ ]:
def test_sample(sample_id):
    global outcome_unknown
    if not server_verified:
        raise RuntimeError('Run the readiness/catalog check successfully first.')
    if outcome_unknown:
        raise RuntimeError('A previous outcome is unknown. Check server completion before starting a new session.')
    if sample_id not in by_id:
        raise ValueError('Choose an ID from the verified catalog.')
    if sample_id in results:
        print(sample_id, ': already requested in this session; skipped.')
        return results[sample_id]
    reference = by_id[sample_id]
    record = {'sample_id': sample_id, 'reference': reference,
              'requested_utc': datetime.now(timezone.utc).isoformat()}
    try:
        status, body = api('/api/demo/' + parse.quote(sample_id, safe='') + '/predict', 'POST')
        record.update(http_status=status, response=body)
        if status == 200:
            p = body['prediction']
            if (body.get('schema_version') != 'retfound-demo-result/1'
                or body['reference']['id'] != sample_id
                or any(body['reference'].get(k) != reference[k] for k in ('sha256', 'true_grade', 'true_label'))
                or p['image_sha256'] != reference['sha256']
                or p['input_validation']['decision'] != 1
                or type(p['predicted_grade']) is not int or p['predicted_grade'] not in range(5)
                or len(p['scores']) != 5 or type(body['matches_true_grade']) is not bool):
                raise ValueError('Unexpected prediction contract or image identity.')
            record['status'] = 'predicted'
        else:
            record['status'] = 'rejected' if status == 422 and body.get('error', {}).get('code') == 'not_retinal_fundus' else 'error'
    except KeyboardInterrupt:
        outcome_unknown = True
        record.update(status='unknown', detail='Client interrupted; server may still be working.')
        results[sample_id] = record
        raise
    except (OSError, ValueError, KeyError, TypeError) as exc:
        # Client failures cannot establish whether the server finished; never retry automatically.
        outcome_unknown = True
        record.update(status='unknown', detail=str(exc))
    results[sample_id] = record
    return record

def show_results():
    rows = []
    for sample in samples:
        r = results.get(sample['id'], {})
        body = r.get('response', {})
        p = body.get('prediction', {}) if r.get('status') == 'predicted' else {}
        gate = 1 if p else (0 if r.get('status') == 'rejected' else 'unknown / not tested')
        rows.append([sample['id'], sample['true_grade'], r.get('status', 'not_tested'), gate,
                     p.get('predicted_grade', ''), body.get('matches_true_grade', '') if p else '',
                     body.get('error', {}).get('message', r.get('detail', ''))])
    table(['Image', 'Reference', 'State', 'Jev', 'DR prediction', 'Backend match', 'Detail'], rows)

if RUN_ONE:
    test_sample(SAMPLE_ID)
    show_results()
    record = results.get(SAMPLE_ID, {})
    if record.get('status') == 'predicted':
        prediction = record['response']['prediction']
        table(['Grade', 'Label', 'Uncalibrated softmax score'],
              [(g, prediction['model']['grades'][g], f'{score:.6f}') for g, score in enumerate(prediction['scores'])])
        print('Request ID:', prediction['request_id'])
else:
    print('Single-image inference disabled. Set RUN_ONE = True to enable.')


## 7. Test all 15 sequentially

Set `RUN_ALL = True` in the settings cell when ready. The loop skips images already requested in this session. It stops on any error or unknown outcome; a definite Jev rejection is recorded and the next image can be tested. There are no automatic retries. A client timeout can leave the server working, so verify server completion before starting a new session.

Interrupting a cell also leaves its current request outcome unknown. Untested images retain `not_tested`; rejected images have no DR grade. A false rejection is not a DR misclassification.


In [ ]:
if RUN_ALL:
    try:
        for sample in samples:
            record = test_sample(sample['id'])
            print(sample['id'], record['status'])
            if record['status'] in ('error', 'unknown'):
                print('Batch stopped. Review the server and the error before making new requests.')
                break
    except KeyboardInterrupt:
        outcome_unknown = True
        if 'sample' in globals() and sample['id'] not in results:
            results[sample['id']] = {'sample_id': sample['id'], 'reference': sample,
                                     'status': 'unknown', 'detail': 'Client interrupted; server may still be working.'}
        print('Interrupted. No further requests sent; current server outcome may be unknown.')
    show_results()
else:
    print('Batch inference disabled. Set RUN_ALL = True to enable.')


## 8. Save your real session

Run the following cell after testing to write a **new** timestamped JSON file under `results/notebook_demo/`. It preserves server responses (checkpoint hash, preprocessing version, actual device, policy version, request IDs and all five scores), reference identities, and explicit missing/error states. Historical outputs are not overwritten. Preview-only sessions write nothing.

The model scores are uncalibrated. A high score is not a verified clinical probability. No aggregate accuracy is computed for this tiny balanced sample.


In [ ]:
if results:
    output_dir = PROJECT_ROOT / 'results/notebook_demo'
    output_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    output_path = output_dir / f'jev-retfound-15-{stamp}-{uuid.uuid4().hex[:8]}.json'
    session = {
        'schema_version': 'retfound-notebook-demo/1',
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'server_url': BASE_URL,
        'manifest_sha256': hashlib.sha256((DEMO_ROOT / 'manifest.json').read_bytes()).hexdigest(),
        'selection': 'Balanced deterministic demonstration; one test image per grade per dataset.',
        'limitations': ['Not representative accuracy evidence', 'Broader split/patient/near-duplicate overlap unverified',
                        'Checkpoint-training independence unverified', 'Uncalibrated scores; research use'],
        'results': [results.get(s['id'], {'sample_id': s['id'], 'reference': s, 'status': 'not_tested'}) for s in samples],
    }
    with output_path.open('x', encoding='utf-8') as file:
        json.dump(session, file, ensure_ascii=False, indent=2, allow_nan=False)
    print('Saved:', output_path)
else:
    print('No inference requested in this session; no results file created.')


## 9. Compare sensitivity and specificity on the saved 1,000-image evaluation

**Sensitivity** measures the proportion of reference DR images detected: **TP / (TP + FN)**.
**Specificity** measures the proportion of reference No DR images correctly classified as No DR: **TN / (TN + FP)**.

For this binary task, reference grades 1–4 are positive and grade 0 is negative. A predicted grade greater than 0 means DR detected. For example, true grade 2 predicted as grade 1 is a binary true positive, although its severity grade is wrong.

This cell recomputes metrics from the saved predictions using the **same shared metric function** as the evaluator. It makes no model/API calls and does not change historical reports. It is independent of the 15-image session above. Undefined denominators display N/A with a reason.

The saved run predates Jev: these values describe RETFound alone. The 15 fundus examples contain no non-fundus negative controls and cannot establish Jev specificity.


In [ ]:
from retfound_platform.evaluation.metrics import metrics

saved_path = PROJECT_ROOT / 'results/runs/aptos1000-20261029/predictions.json'
saved_evaluation = json.loads(saved_path.read_text(encoding='utf-8'))
evaluated_metrics = metrics(saved_evaluation['records'])
binary_metrics = evaluated_metrics['binary']

def metric_percent(value):
    return 'N/A' if value is None else f'{100 * value:.2f}%'

table(['Measure', 'Value', 'Meaning'], [
    ['Images evaluated', evaluated_metrics['n'], 'Saved APTOS evaluation; RETFound alone'],
    ['Exact five-grade accuracy', metric_percent(evaluated_metrics['accuracy']), 'Correct severity grade'],
    ['Any-DR sensitivity', metric_percent(binary_metrics['sensitivity']), 'Reference DR cases detected'],
    ['Any-DR specificity', metric_percent(binary_metrics['specificity']), 'Reference No DR cases correctly excluded'],
])
table(['Count', 'Value', 'Meaning'], [
    ['TP', binary_metrics['tp'], 'Reference DR; predicted DR'],
    ['FN', binary_metrics['fn'], 'Reference DR; predicted No DR (missed cases)'],
    ['TN', binary_metrics['tn'], 'Reference No DR; predicted No DR'],
    ['FP', binary_metrics['fp'], 'Reference No DR; predicted DR (false alarms)'],
])
for metric_name, numerator, other in [('sensitivity', 'tp', 'fn'), ('specificity', 'tn', 'fp')]:
    value = binary_metrics[metric_name]
    if value is None:
        print(metric_name.title() + ': N/A. ' + evaluated_metrics['undefined_reasons'][metric_name])
    else:
        print(f'{metric_name.title()}: {binary_metrics[numerator]} / ({binary_metrics[numerator]} + {binary_metrics[other]}) = {metric_percent(value)}')
print('Run:', saved_evaluation['run_id'], '| No new inference performed.')


## 10. Interpret the test and troubleshoot

- **Jev 1 + DR prediction:** accepted modality; compare its DR grade with the reference using the server's `matches_true_grade` field.
- **Jev 0:** rejected before DR inference. Investigate rejection separately; these 15 samples contain no non-fundus negative controls, so this set cannot measure gate specificity.
- **HTTP 400 / 503:** review the backend diagnostic, API key, credits, endpoint and connection. The implemented model is `typesafe/jev-1.14`; backend POST endpoint is `https://openrouter.ai/api/alpha/decisions`. This is an ordinary authenticated HTTP integration, with the Jev schema, rather than a workflow node. Do not substitute a chat-completions payload.
- **HTTP 409:** image bytes changed relative to the manifest; no inference is run for that sample.
- **HTTP 429:** server is busy. This notebook stops and does not automatically repeat the request.
- **Unknown / timeout / interrupted:** the client cannot determine whether server inference completed. Check the server before beginning another session.
- **No server:** local previews still work. Start the service, wait for readiness and rerun section 5.

The saved **1,000-image evaluation** in `results/runs/aptos1000-20261029` predates Jev and evaluates RETFound alone. It is separate from this 15-image demonstration. Do not describe it as a 1,000-image Jev + RETFound validation.

Further reading: `docs/API.md`, `docs/PROGRESS.md`, `upload_test_images/huggingface15-20261030/README.md` and `docs/reports/retfound-jev-20261030/RETFound_Jev_Technical_Report.md`.
